# Week 02 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 02 Day 01: Random variables and key distributions

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Strengthen statistical reasoning needed for quant inference and model validation.

## Continuity and Handoff
- Previous checkpoint: Week 01 Day 07: Portfolio Project
- Previous lesson file: content/week-01/day-07.md
- Today's deliverable: Simulate returns from multiple distributions and compare tail events.
- Next handoff target: Week 02 Day 02: Expectation, variance, and covariance
- Next lesson file: content/week-02/day-02.md

## Theory Concepts

### Concept 1: Discrete vs continuous random variables
Discrete vs continuous random variables is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Normal, lognormal, and heavy-tail intuition
Normal, lognormal, and heavy-tail intuition is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Distribution moments and shape
Distribution moments and shape is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Compare simulated normal returns with fat-tail synthetic returns.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Random variables and key distributions'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $102.00 to $103.22.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (103.224-102.000)/102.000 = 0.012000.
Final answer: Simple return = 1.20%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0112.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0112 = 0.1778.
Final answer: Annualized volatility = 17.78%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 17.78%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.1778 = 0.6187.
Final answer: Sharpe ratio = 0.62.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Simulate returns from multiple distributions and compare tail events.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 02 Day 01): Explain Simple Return and define every symbol clearly.
   - Model answer: "I use Simple Return to convert raw prices into a decision-ready metric. The formula is $r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Random variables and key distributions' matter in live trading systems?
   - Model answer: "Random variables and key distributions matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Random variables and key distributions as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
Why are heavy tails critical for risk systems?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 02 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(201)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_60355/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019171355226737,
 'annualized_return': 0.1422348002525946,
 'annualized_volatility': 0.19307149681123456,
 'max_drawdown': -0.337172402602658,
 'spy_rate_corr': 0.008052788068407915}

## Week 02 Day 01 Quiz

Topic: **Random variables and key distributions**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [3]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 103.000
price_t = 103.875
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Random variables and key distributions')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.008495)
print('  gross_return_expected  =', 1.008495)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 103.0
  P_t    : 103.875
  r_t    : 0.008495 => 0.85%
  1+r_t  : 1.008495

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Random variables and key distributions
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.008495
  gross_return_expected  = 1.008495


# Week 02 Day 02: Expectation, variance, and covariance

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Strengthen statistical reasoning needed for quant inference and model validation.

## Continuity and Handoff
- Previous checkpoint: Week 02 Day 01: Random variables and key distributions
- Previous lesson file: content/week-02/day-01.md
- Today's deliverable: Write a covariance diagnostics function with pairwise interpretation.
- Next handoff target: Week 02 Day 03: Sampling and estimation
- Next lesson file: content/week-02/day-03.md

## Theory Concepts

### Concept 1: Expected value as long-run average
Expected value as long-run average is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Variance as uncertainty proxy
Variance as uncertainty proxy is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Covariance/correlation for co-movement
Covariance/correlation for co-movement is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Estimate covariance matrix for a small asset universe.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Expectation, variance, and covariance'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $104.00 to $105.35.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (105.352-104.000)/104.000 = 0.013000.
Final answer: Simple return = 1.30%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0119.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0119 = 0.1889.
Final answer: Annualized volatility = 18.89%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 18.89%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.1889 = 0.5823.
Final answer: Sharpe ratio = 0.58.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Write a covariance diagnostics function with pairwise interpretation.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 02 Day 02): Explain Log Return and define every symbol clearly.
   - Model answer: "I use Log Return to convert raw prices into a decision-ready metric. The formula is $\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Expectation, variance, and covariance' matter in live trading systems?
   - Model answer: "Expectation, variance, and covariance matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Expectation, variance, and covariance as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
What can high correlation hide during market stress?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 02 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [4]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(202)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_60355/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.00191694189232,
 'annualized_return': 0.14223479133923544,
 'annualized_volatility': 0.19307141174237197,
 'max_drawdown': -0.3371728444752473,
 'spy_rate_corr': 0.008052801920852828}

## Week 02 Day 02 Quiz

Topic: **Expectation, variance, and covariance**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [5]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 104.000
price_t = 104.936
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Expectation, variance, and covariance')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009000)
print('  gross_return_expected  =', 1.009000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 104.0
  P_t    : 104.936
  r_t    : 0.009 => 0.90%
  1+r_t  : 1.009

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Expectation, variance, and covariance
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.009
  gross_return_expected  = 1.009


# Week 02 Day 03: Sampling and estimation

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Strengthen statistical reasoning needed for quant inference and model validation.

## Continuity and Handoff
- Previous checkpoint: Week 02 Day 02: Expectation, variance, and covariance
- Previous lesson file: content/week-02/day-02.md
- Today's deliverable: Bootstrap sample means and visualize confidence spread.
- Next handoff target: Week 02 Day 04: Confidence intervals and hypothesis testing
- Next lesson file: content/week-02/day-04.md

## Theory Concepts

### Concept 1: Population vs sample
Population vs sample is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Estimator bias and variance tradeoff
Estimator bias and variance tradeoff is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Central limit theorem intuition
Central limit theorem intuition is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Estimate mean return stability across different sample lengths.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Sampling and estimation'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $106.00 to $107.48.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (107.484-106.000)/106.000 = 0.014000.
Final answer: Simple return = 1.40%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0126.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0126 = 0.2000.
Final answer: Annualized volatility = 20.00%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 20.00%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2000 = 0.5499.
Final answer: Sharpe ratio = 0.55.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Bootstrap sample means and visualize confidence spread.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 02 Day 03): Explain Annualized Volatility and define every symbol clearly.
   - Model answer: "I use Annualized Volatility to convert raw prices into a decision-ready metric. The formula is $\sigma_{ann}=\sqrt{252}\,Std(r_t)$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Sampling and estimation' matter in live trading systems?
   - Model answer: "Sampling and estimation matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Sampling and estimation as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
How does small sample size distort confidence?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 02 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [6]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(203)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_60355/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019177164139315,
 'annualized_return': 0.1422348269926783,
 'annualized_volatility': 0.19307156631578165,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052772347549867}

## Week 02 Day 03 Quiz

Topic: **Sampling and estimation**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [7]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 105.000
price_t = 105.998
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Sampling and estimation')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009505)
print('  gross_return_expected  =', 1.009505)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 105.0
  P_t    : 105.998
  r_t    : 0.009505 => 0.95%
  1+r_t  : 1.009505

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Sampling and estimation
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.009505
  gross_return_expected  = 1.009505


# Week 02 Day 04: Confidence intervals and hypothesis testing

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Strengthen statistical reasoning needed for quant inference and model validation.

## Continuity and Handoff
- Previous checkpoint: Week 02 Day 03: Sampling and estimation
- Previous lesson file: content/week-02/day-03.md
- Today's deliverable: Implement one-sided and two-sided tests on synthetic strategy returns.
- Next handoff target: Week 02 Day 05: Statistical communication for finance decisions
- Next lesson file: content/week-02/day-05.md

## Theory Concepts

### Concept 1: Confidence intervals for means
Confidence intervals for means is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Null/alternative hypotheses
Null/alternative hypotheses is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: P-value interpretation pitfalls
P-value interpretation pitfalls is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Test whether strategy mean return differs from zero.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Confidence intervals and hypothesis testing'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $108.00 to $109.62.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (109.620-108.000)/108.000 = 0.015000.
Final answer: Simple return = 1.50%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0133.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0133 = 0.2111.
Final answer: Annualized volatility = 21.11%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 21.11%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2111 = 0.5210.
Final answer: Sharpe ratio = 0.52.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement one-sided and two-sided tests on synthetic strategy returns.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 02 Day 04): Explain Sharpe Ratio and define every symbol clearly.
   - Model answer: "I use Sharpe Ratio to convert raw prices into a decision-ready metric. The formula is $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Confidence intervals and hypothesis testing' matter in live trading systems?
   - Model answer: "Confidence intervals and hypothesis testing matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Confidence intervals and hypothesis testing as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
Why is statistical significance not equal to economic significance?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 02 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(204)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_60355/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019171355226684,
 'annualized_return': 0.14223480025259438,
 'annualized_volatility': 0.19307150548953592,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052787945709749}

## Week 02 Day 04 Quiz

Topic: **Confidence intervals and hypothesis testing**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [9]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 106.000
price_t = 107.060
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Confidence intervals and hypothesis testing')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010000)
print('  gross_return_expected  =', 1.010000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 106.0
  P_t    : 107.06
  r_t    : 0.01 => 1.00%
  1+r_t  : 1.01

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Confidence intervals and hypothesis testing
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.01
  gross_return_expected  = 1.01


# Week 02 Day 05: Statistical communication for finance decisions

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Strengthen statistical reasoning needed for quant inference and model validation.

## Continuity and Handoff
- Previous checkpoint: Week 02 Day 04: Confidence intervals and hypothesis testing
- Previous lesson file: content/week-02/day-04.md
- Today's deliverable: Create a short decision report generator from test outputs.
- Next handoff target: Week 02 Day 06: Revision Sprint
- Next lesson file: content/week-02/day-06.md

## Theory Concepts

### Concept 1: Effect size versus significance
Effect size versus significance is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Practical significance under costs
Practical significance under costs is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Decision thresholds under uncertainty
Decision thresholds under uncertainty is a core part of 'Probability and statistics for market data'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Translate a test result into a go/no-go trading research decision.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Statistical communication for finance decisions'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $110.00 to $111.76.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (111.760-110.000)/110.000 = 0.016000.
Final answer: Simple return = 1.60%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0140.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0140 = 0.2222.
Final answer: Annualized volatility = 22.22%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 22.22%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2222 = 0.4950.
Final answer: Sharpe ratio = 0.49.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create a short decision report generator from test outputs.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 02 Day 05): Explain Turnover and define every symbol clearly.
   - Model answer: "I use Turnover to convert raw prices into a decision-ready metric. The formula is $TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Statistical communication for finance decisions' matter in live trading systems?
   - Model answer: "Statistical communication for finance decisions matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Statistical communication for finance decisions as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
How would you explain uncertainty to a non-technical stakeholder?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 02 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [10]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(205)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_60355/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019171355226737,
 'annualized_return': 0.1422348002525946,
 'annualized_volatility': 0.19307149681123456,
 'max_drawdown': -0.337172402602658,
 'spy_rate_corr': 0.008052788068407915}

## Week 02 Day 05 Quiz

Topic: **Statistical communication for finance decisions**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [11]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 107.000
price_t = 108.123
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Statistical communication for finance decisions')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010495)
print('  gross_return_expected  =', 1.010495)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 107.0
  P_t    : 108.123
  r_t    : 0.010495 => 1.05%
  1+r_t  : 1.010495

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Statistical communication for finance decisions
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.010495
  gross_return_expected  = 1.010495


# Week 02 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 02 Day 05: Statistical communication for finance decisions
- Previous lesson file: content/week-02/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 02 Day 07: Portfolio Project
- Next lesson file: content/week-02/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Revisit CLT, confidence intervals, and testing assumptions
- Redo one bootstrap exercise without notes
- Document top 3 interpretation mistakes in error log

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 02 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 02 Day 06: Revision Sprint
- Previous lesson file: content/week-02/day-06.md
- Today's deliverable: Bootstrap confidence interval study
- Next handoff target: Week 03 Day 01: Vectors, matrices, and transformations
- Next lesson file: content/week-03/day-01.md

## Project Title
Bootstrap confidence interval study

## Problem Statement
Estimate uncertainty around average returns for multiple assets using resampling.

## Data Sources
- Yahoo Finance
- FRED risk-free proxy
- Synthetic stress scenarios

## Implementation Steps
1. Define estimation target
2. Build bootstrap sampler
3. Compute interval estimates
4. Compare stable vs volatile assets
5. Summarize implications for portfolio design

## Evaluation Metrics
- Interval width
- Coverage intuition
- Sampling stability
- Decision confidence

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
